In [2]:
# Notebook 03 - Feature Engineering (step-by-step)

# Cell A - Setup & Config
# - Import libs: pandas, numpy, ast, sklearn, scipy.sparse, pathlib, json, joblib
# - Define config: input/output paths, seed, TF-IDF params, topk_similarity

# Cell B - Load processed data
# - Load data/processed/recipes_clean.csv
# - Load data/processed/interactions_clean.csv
# - Parse list-like columns: ingredients_clean, tags_clean

# Cell C - Schema checks & hard guards
# - Validate dtypes: id, user_id, recipe_id, rating, minutes, calories
# - Remove orphan interactions (recipe_id not in recipes)
# - Enforce feature ranges (minutes > 0, calories in [10, 5000] or flag outlier)

# Cell D - Anti-leakage split
# - Split interactions: train/validation (time-based or leave-last-1 per user)
# - Use train_interactions to compute user/popularity statistics
# - Do not use validation interactions when fitting user-level stats

# Cell E - Text feature construction
# - Build ingredients_text, tags_text, description_clean
# - Build combined_text with deterministic weighting strategy

# Cell F - TF-IDF recipe vectors
# - Fit TfidfVectorizer on combined_text
# - Build recipe_tfidf_matrix + recipe_id/index mapping

# Cell G - Content similarity artifacts
# - Compute top-k cosine neighbors per recipe (avoid full NxN persistence)
# - Output columns: recipe_id, neighbor_recipe_id, similarity

# Cell H - Numerical & constraint features
# - Build rule-based fields: calories, minutes, n_ingredients_calc
# - Build buckets/flags: quick_meal, low_calorie, etc.

# Cell I - User features (from train only)
# - avg_rating_given, rating_count, activity_level
# - favorite_tags from high-rated history
# - default fallback for new users

# Cell J - Recipe popularity features
# - rating_count, rating_mean, popularity_score (Bayesian smoothing)
# - is_cold_item flag for cold-start logic

# Cell K - Feature validation checklist
# - Validate nulls, schema, key uniqueness
# - Coverage checks: users/features, recipes/vectors, recipes/top-k neighbors
# - Print final sanity summary

# Cell L - Save artifacts + manifest
# - Save into artifacts/features/
#   tfidf_vectorizer.joblib
#   recipe_tfidf_matrix.npz
#   recipe_index_map.parquet
#   recipe_similarity_topk.parquet
#   recipe_features.parquet
#   user_features.parquet
#   popularity_features.parquet
#   feature_manifest.json

# Output
# - Print artifact paths, shapes, coverage, and build timestamp

In [3]:
import pandas as pd
import numpy as np
import re
import ast
import json
import joblib
import time
from pathlib import Path
from datetime import datetime
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [4]:
df_recipes = pd.read_csv("../data/processed/recipes_clean.csv")
df_interactions = pd.read_csv("../data/processed/interactions_clean.csv")

def parse_list_safe(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x.strip())
            return v if isinstance(v, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

for col in ["ingredients_clean", "tags_clean", "ingredients", "tags", "steps"]:
    if col in df_recipes.columns:
        df_recipes[col] = df_recipes[col].apply(parse_list_safe)

print(f"recipes:      {df_recipes.shape}")
print(f"interactions: {df_interactions.shape}")
print(f"Sample ingredients_clean: {df_recipes['ingredients_clean'].iloc[0][:3]}")
print(f"Sample tags_clean:        {df_recipes['tags_clean'].iloc[0][:3]}")


recipes:      (77300, 24)
interactions: (706514, 5)
Sample ingredients_clean: ['winter squash', 'mexican seasoning', 'mixed spice']
Sample tags_clean:        ['60-minutes-or-less', 'time-to-make', 'course']


In [5]:
# === Cell C - Schema checks & hard guards ===

print("--- Before guards ---")
print(f"recipes:      {df_recipes.shape}")
print(f"interactions: {df_interactions.shape}")

# 1) Ép kiểu dữ liệu core
df_recipes["id"] = pd.to_numeric(df_recipes["id"], errors="coerce").astype("Int64")
df_recipes["minutes"] = pd.to_numeric(df_recipes["minutes"], errors="coerce")
df_recipes["calories"] = pd.to_numeric(df_recipes["calories"], errors="coerce")

df_interactions["user_id"] = pd.to_numeric(df_interactions["user_id"], errors="coerce").astype("Int64")
df_interactions["recipe_id"] = pd.to_numeric(df_interactions["recipe_id"], errors="coerce").astype("Int64")
df_interactions["rating"] = pd.to_numeric(df_interactions["rating"], errors="coerce")
if "date" in df_interactions.columns:
    df_interactions["date"] = pd.to_datetime(df_interactions["date"], errors="coerce")

# 2) Loại orphan interactions (recipe_id không có trong recipes)
valid_ids = set(df_recipes["id"].dropna())
n_before = len(df_interactions)
df_interactions = df_interactions[df_interactions["recipe_id"].isin(valid_ids)].copy()
print(f"Orphan interactions removed: {n_before - len(df_interactions):,}")

# 3) Filter range hợp lý cho feature
df_recipes = df_recipes[df_recipes["minutes"].between(1, 1440)].copy()
df_recipes["is_calorie_outlier"] = ~df_recipes["calories"].between(10, 5000)
n_outlier = df_recipes["is_calorie_outlier"].sum()
print(f"Calorie outliers flagged (kept but flagged): {n_outlier:,}")

# 4) Sync lại sau filter
df_interactions = df_interactions[df_interactions["recipe_id"].isin(set(df_recipes["id"]))].copy()

print("\n--- After guards ---")
print(f"recipes:      {df_recipes.shape}")
print(f"interactions: {df_interactions.shape}")
print(f"unique users: {df_interactions['user_id'].nunique():,}")
print(f"unique items: {df_interactions['recipe_id'].nunique():,}")
print(f"orphan check: {len(set(df_interactions['recipe_id']) - set(df_recipes['id']))} orphans")

--- Before guards ---
recipes:      (77300, 24)
interactions: (706514, 5)
Orphan interactions removed: 7,138
Calorie outliers flagged (kept but flagged): 692

--- After guards ---
recipes:      (77300, 25)
interactions: (699376, 5)
unique users: 30,836
unique items: 77,300
orphan check: 0 orphans


In [6]:
# === Cell D - Anti-leakage split ===
# Split interactions theo thời gian: 80% cũ nhất -> train, 20% mới nhất -> validation
# Chỉ dùng train_interactions để tính user/popularity stats phía sau

interactions_sorted = df_interactions.sort_values("date").reset_index(drop=True)
split_idx = int(len(interactions_sorted) * 0.8)

train_interactions = interactions_sorted.iloc[:split_idx].copy()
val_interactions = interactions_sorted.iloc[split_idx:].copy()

train_user_ids = set(train_interactions["user_id"])
train_recipe_ids = set(train_interactions["recipe_id"])
val_user_ids = set(val_interactions["user_id"])
val_recipe_ids = set(val_interactions["recipe_id"])

cold_start_users = val_user_ids - train_user_ids
cold_start_items = val_recipe_ids - train_recipe_ids

print(f"train_interactions: {len(train_interactions):,} rows")
print(f"val_interactions:   {len(val_interactions):,} rows")
print(f"train date range:   {train_interactions['date'].min()} -> {train_interactions['date'].max()}")
print(f"val   date range:   {val_interactions['date'].min()} -> {val_interactions['date'].max()}")
print(f"\ntrain users: {len(train_user_ids):,}  |  train items: {len(train_recipe_ids):,}")
print(f"val   users: {len(val_user_ids):,}  |  val   items: {len(val_recipe_ids):,}")
print(f"cold-start users in val: {len(cold_start_users):,}")
print(f"cold-start items in val: {len(cold_start_items):,}")

train_interactions: 559,500 rows
val_interactions:   139,876 rows
train date range:   2000-02-25 00:00:00 -> 2010-08-14 00:00:00
val   date range:   2010-08-14 00:00:00 -> 2018-12-19 00:00:00

train users: 27,021  |  train items: 72,860
val   users: 14,375  |  val   items: 44,122
cold-start users in val: 3,815
cold-start items in val: 4,440


In [7]:
# === Cell E ===
# plan Cell E : Xác định các nguồn text trong dataset --> NormalizeText --> Convert list --> Tạo text
def normalize_text(text):
    """
    Normalize general text fields (description)
    """
    if pd.isna(text):
        return ""
    
    text = text.lower()
    
    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def normalize_ingredient(token):
    """
    Normalize ingredient tokens
    """
    if pd.isna(token):
        return ""
    
    token = token.lower().strip()
    
    # replace spaces with underscore
    token = token.replace(" ", "_")
    
    return token


def list_to_text(lst):
    """
    Convert list column to normalized text
    """
    if not isinstance(lst, list):
        return ""
    
    tokens = [normalize_ingredient(x) for x in lst]
    
    return " ".join(tokens)


def tags_to_text(lst):
    """
    Convert tags list to normalized text
    """
    if not isinstance(lst, list):
        return ""
    
    tokens = [normalize_text(x) for x in lst]
    
    return " ".join(tokens)


# ---------- Build text features ----------

# ingredients text
df_recipes["ingredients_text"] = df_recipes["ingredients_clean"].apply(list_to_text)

# tags text
df_recipes["tags_text"] = df_recipes["tags_clean"].apply(tags_to_text)

# description text
if "description" in df_recipes.columns:
    df_recipes["description_clean"] = df_recipes["description"].apply(normalize_text)
else:
    df_recipes["description_clean"] = ""


# ---------- Build combined_text with weighting ----------

def build_combined_text(row):
    
    ingredients = row["ingredients_text"]
    tags = row["tags_text"]
    description = row["description_clean"]
    
    combined = " ".join([
        ingredients,        # ingredients weight 1
        ingredients,        # ingredients weight 2
        tags,               # tags weight 1
        tags,               # tags weight 2
        description         # description weight 1
    ])
    
    return combined


df_recipes["combined_text"] = df_recipes.apply(build_combined_text, axis=1)


# ---------- Basic checks ----------

print("Text feature construction completed\n")

print("Example rows:\n")
display(
    df_recipes[
        ["id","ingredients_text","tags_text","description_clean","combined_text"]
    ].head()
)

print("\nNull counts:")
print(df_recipes[["ingredients_text","tags_text","combined_text"]].isnull().sum())

print("\nAverage text length:")
print(df_recipes["combined_text"].str.len().mean())

Text feature construction completed

Example rows:



,id,ingredients_text,tags_text,description_clean,combined_text
0,137739,winter_squash mexican_seasoning mixed_spice ho...,60 minutes or less time to make course main in...,autumn is my favorite time of year to cook thi...,winter_squash mexican_seasoning mixed_spice ho...
1,75452,sugar unsalted_butter bananas eggs fresh_lemon...,weeknight time to make course main ingredient ...,from ann hodgman s,sugar unsalted_butter bananas eggs fresh_lemon...
2,63986,lean_pork_chops flour salt dry_mustard garlic_...,weeknight time to make course main ingredient ...,here s and old standby i enjoy from time to ti...,lean_pork_chops flour salt dry_mustard garlic_...
3,43026,egg_roll_wrap whole_green_chilies cheese corns...,60 minutes or less time to make course main in...,a favorite from a local restaurant no longer i...,egg_roll_wrap whole_green_chilies cheese corns...
4,23933,butterscotch_chips chinese_noodles salted_peanuts,15 minutes or less time to make course prepara...,a little different and oh so good i include th...,butterscotch_chips chinese_noodles salted_pean...



Null counts:
ingredients_text    0
tags_text           0
combined_text       0
dtype: int64

Average text length:
844.5295601552393


In [8]:
print(df_recipes["combined_text"].isnull().sum())
print(df_recipes["combined_text"].sample(5))
print(df_recipes["combined_text"].str.split().explode().value_counts().head(10))

0
3513     white_sugar cocoa milk butter light_corn_syrup...
69155    swiss_chard olive_oil onions garlic_cloves red...
55115    pork_butt water chicken_bouillon_powder diced_...
41752    acorn_squash water sugar lemon_juice butter sa...
49110    lean_ground_beef egg milk carrot red_bell_pepp...
Name: combined_text, dtype: str
combined_text
low            248594
or             239368
to             230029
less           201544
make           172018
main           169126
time           165680
preparation    155027
course         147578
and            121087
Name: count, dtype: int64


In [9]:
# === Cell F - TF-IDF Recipe Vectors ===
#
# Mục tiêu: biến combined_text của mỗi recipe thành vector số để tính similarity.
#
# Pipeline:
#   combined_text (77k strings)
#       -> TfidfVectorizer.fit_transform()
#           -> tfidf_vectorizer : model đã học vocabulary (token -> column index)
#           -> recipe_tfidf_matrix : sparse matrix (n_recipes x n_features), mỗi hàng = 1 recipe vector
#       -> recipe_index_map : bảng ánh xạ recipe_id <-> row index trong matrix
#
# Tham số TF-IDF:
#   max_features=20000  : giới hạn vocab size, giữ top 20k token theo tần suất
#   ngram_range=(1,2)   : bắt cả unigram (chicken) lẫn bigram (chicken_breast)
#   min_df=5            : bỏ token xuất hiện < 5 recipes (quá hiếm, không khái quát)
#   max_df=0.95         : bỏ token xuất hiện > 95% recipes (stopword-like: "low", "to", "or")
#   sublinear_tf=True   : dùng 1+log(tf) thay raw count, giảm bias cho text dài

tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
)

recipe_tfidf_matrix = tfidf_vectorizer.fit_transform(df_recipes["combined_text"])

# Ánh xạ recipe_id <-> vị trí hàng trong matrix, dùng để tra cứu ở Cell G và notebook 04
recipe_index_map = pd.DataFrame({
    "recipe_id": df_recipes["id"].values,
    "matrix_index": range(len(df_recipes))
})

# ---------- Sanity check ----------
print(f"TF-IDF matrix shape: {recipe_tfidf_matrix.shape}")
print(f"Vocabulary size:     {len(tfidf_vectorizer.vocabulary_)}")
print(f"Sparsity:            {1 - recipe_tfidf_matrix.nnz / (recipe_tfidf_matrix.shape[0] * recipe_tfidf_matrix.shape[1]):.4%}")

vocab = tfidf_vectorizer.get_feature_names_out()
print(f"\nSample vocab (first 10):  {list(vocab[:10])}")
print(f"Sample vocab (last 10):   {list(vocab[-10:])}")
print(f"\nrecipe_index_map:")
display(recipe_index_map.head())


TF-IDF matrix shape: (77300, 20000)
Vocabulary size:     20000
Sparsity:            99.4729%

Sample vocab (first 10):  ['05', '06', '07', '08', '09', '10', '10 minutes', '10 years', '100', '11']
Sample vocab (last 10):   ['zucchini red_bell_pepper', 'zucchini salt', 'zucchini tomatoes', 'zucchini yellow_squash', 'zwt', 'zwt region', 'zwt3', 'zwt5', 'zwt6', 'zwt7']

recipe_index_map:


,recipe_id,matrix_index
0,137739,0
1,75452,1
2,63986,2
3,43026,3
4,23933,4


In [10]:
# === Cell G - Content Similarity Artifacts ===
#
# Mục tiêu: Tính top-K cosine neighbors cho mỗi recipe từ TF-IDF matrix.
#
# Tại sao không tính full NxN?
#   - 77k × 77k = ~6 tỷ giá trị → ~45GB RAM → không khả thi
#   - Chỉ cần giữ top-K neighbors là đủ cho recommendation
#
# Chiến lược batch:
#   1. Chia 77k recipes thành batch nhỏ (BATCH_SIZE=500)
#   2. Với mỗi batch, tính cosine_similarity(batch, full_matrix)
#      → ra sub_matrix (500 × 77k)
#   3. Lấy top-K indices + scores từ mỗi hàng
#   4. Gom kết quả vào list, cuối cùng concat thành DataFrame
#
# Output: recipe_similarity_topk  (DataFrame)
#   Columns: recipe_id, neighbor_recipe_id, similarity
#   Rows:    n_recipes × TOP_K (≈ 77k × 50 ≈ 3.8M rows)

TOP_K = 50           # số neighbors giữ lại mỗi recipe
BATCH_SIZE = 500     # số recipes xử lý mỗi vòng lặp
n_recipes = recipe_tfidf_matrix.shape[0]
all_recipe_ids = recipe_index_map["recipe_id"].values   # array recipe_id theo thứ tự hàng matrix
print(f"Computing top-{TOP_K} neighbors for {n_recipes:,} recipes (batch_size={BATCH_SIZE})")
print(f"Expected output rows: ~{n_recipes * TOP_K:,}")
t0 = time.time()
results = []   # list of (recipe_id, neighbor_recipe_id, similarity)
for start in range(0, n_recipes, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n_recipes)
    # --- 1. Tính cosine similarity cho batch này với toàn bộ matrix ---
    # batch_matrix: (batch_size, 20000) — sparse
    # recipe_tfidf_matrix: (77300, 20000) — sparse
    # sim_matrix: (batch_size, 77300) — dense ndarray
    sim_matrix = cosine_similarity(
        recipe_tfidf_matrix[start:end],    # batch rows
        recipe_tfidf_matrix                 # full matrix
    )
    # --- 2. Với mỗi recipe trong batch, lấy top-K neighbors ---
    for local_idx in range(sim_matrix.shape[0]):
        global_idx = start + local_idx
        scores = sim_matrix[local_idx]        # (77300,)
        # Đặt self-similarity = -1 để loại chính nó ra khỏi top-K
        scores[global_idx] = -1.0
        # np.argpartition nhanh hơn np.argsort cho top-K (O(n) vs O(n log n))
        # Lấy K indices có score cao nhất (chưa sắp xếp)
        top_k_unsorted = np.argpartition(scores, -TOP_K)[-TOP_K:]
        # Sắp xếp lại K indices theo score giảm dần
        top_k_sorted = top_k_unsorted[np.argsort(scores[top_k_unsorted])[::-1]]
        src_id = all_recipe_ids[global_idx]
        for neighbor_idx in top_k_sorted:
            results.append((
                src_id,
                all_recipe_ids[neighbor_idx],
                float(scores[neighbor_idx])
            ))
    # Progress log mỗi 5000 recipes
    if (end % 5000 == 0) or (end == n_recipes):
        elapsed = time.time() - t0
        pct = end / n_recipes * 100
        print(f"  [{pct:5.1f}%] {end:,}/{n_recipes:,} recipes done  ({elapsed:.1f}s)")
# --- 3. Tạo DataFrame kết quả ---
recipe_similarity_topk = pd.DataFrame(
    results,
    columns=["recipe_id", "neighbor_recipe_id", "similarity"]
)
elapsed_total = time.time() - t0
print(f"\nDone in {elapsed_total:.1f}s")
print(f"Output shape: {recipe_similarity_topk.shape}")
print(f"Similarity range: [{recipe_similarity_topk['similarity'].min():.4f}, "
      f"{recipe_similarity_topk['similarity'].max():.4f}]")
print(f"Mean similarity:  {recipe_similarity_topk['similarity'].mean():.4f}")
# --- 4. Quick sanity check ---
# Lấy 1 recipe mẫu, xem top-5 neighbors có hợp lý không
sample_id = all_recipe_ids[0]
sample_neighbors = recipe_similarity_topk[
    recipe_similarity_topk["recipe_id"] == sample_id
].head(5)
print(f"\nSample: top-5 neighbors of recipe_id={sample_id}")
print(f"  Recipe name: {df_recipes.loc[df_recipes['id'] == sample_id, 'name'].values[0]}")
for _, row in sample_neighbors.iterrows():
    nb_name = df_recipes.loc[df_recipes["id"] == row["neighbor_recipe_id"], "name"].values
    nb_name = nb_name[0] if len(nb_name) > 0 else "?"
    print(f"  → {row['neighbor_recipe_id']}  sim={row['similarity']:.4f}  {nb_name}")


Computing top-50 neighbors for 77,300 recipes (batch_size=500)
Expected output rows: ~3,865,000
  [  6.5%] 5,000/77,300 recipes done  (18.3s)
  [ 12.9%] 10,000/77,300 recipes done  (33.2s)
  [ 19.4%] 15,000/77,300 recipes done  (47.7s)
  [ 25.9%] 20,000/77,300 recipes done  (61.6s)
  [ 32.3%] 25,000/77,300 recipes done  (76.3s)
  [ 38.8%] 30,000/77,300 recipes done  (92.5s)
  [ 45.3%] 35,000/77,300 recipes done  (106.6s)
  [ 51.7%] 40,000/77,300 recipes done  (121.1s)
  [ 58.2%] 45,000/77,300 recipes done  (134.9s)
  [ 64.7%] 50,000/77,300 recipes done  (148.7s)
  [ 71.2%] 55,000/77,300 recipes done  (163.7s)
  [ 77.6%] 60,000/77,300 recipes done  (184.5s)
  [ 84.1%] 65,000/77,300 recipes done  (199.0s)
  [ 90.6%] 70,000/77,300 recipes done  (213.0s)
  [ 97.0%] 75,000/77,300 recipes done  (227.0s)
  [100.0%] 77,300/77,300 recipes done  (233.6s)

Done in 236.8s
Output shape: (3865000, 3)
Similarity range: [0.0948, 0.9919]
Mean similarity:  0.2768

Sample: top-5 neighbors of recipe_id=13

In [11]:
# Cell H - Numerical & constraint features
# - Build rule-based fields: calories, minutes, n_ingredients_calc
# - Build buckets/flags: quick_meal, low_calorie, etc.
# Thời gian nấu
df_recipes["is_quick_meal"] = df_recipes["minutes"] <= 30
df_recipes["is_medium_meal"] = df_recipes["minutes"].between(31, 60)
df_recipes["is_long_meal"] = df_recipes["minutes"] > 60

# Calories
df_recipes["is_low_calorie"] = df_recipes["calories"].between(10, 300)
df_recipes["is_medium_calorie"] = df_recipes["calories"].between(301, 600)
df_recipes["is_high_calorie"] = df_recipes["calories"] > 600

# Độ phức tạp theo số nguyên liệu
df_recipes["is_simple_recipe"] = df_recipes["n_ingredients_calc"] <= 5
df_recipes["is_complex_recipe"] = df_recipes["n_ingredients_calc"] >= 15

recipe_features = df_recipes[[
    "id", "minutes", "calories", "n_ingredients_calc",
    "is_calorie_outlier",
    "is_quick_meal", "is_medium_meal", "is_long_meal",
    "is_low_calorie", "is_medium_calorie", "is_high_calorie",
    "is_simple_recipe", "is_complex_recipe"
]].copy()
recipe_features = recipe_features.rename(columns={"id": "recipe_id"})

print(f"recipe_features shape: {recipe_features.shape}")
print(f"\nDistribution:")
print(f"  quick_meal:    {recipe_features['is_quick_meal'].sum():,} ({recipe_features['is_quick_meal'].mean():.1%})")
print(f"  low_calorie:   {recipe_features['is_low_calorie'].sum():,} ({recipe_features['is_low_calorie'].mean():.1%})")
print(f"  simple_recipe: {recipe_features['is_simple_recipe'].sum():,} ({recipe_features['is_simple_recipe'].mean():.1%})")
print(f"\nSample:")
display(recipe_features.head())

recipe_features shape: (77300, 13)

Distribution:
  quick_meal:    34,877 (45.1%)
  low_calorie:   38,104 (49.3%)
  simple_recipe: 14,251 (18.4%)

Sample:


,recipe_id,minutes,calories,n_ingredients_calc,is_calorie_outlier,is_quick_meal,is_medium_meal,is_long_meal,is_low_calorie,is_medium_calorie,is_high_calorie,is_simple_recipe,is_complex_recipe
0,137739,55,51.5,7,False,False,True,False,True,False,False,False,False
1,75452,70,2669.3,9,False,False,False,True,False,False,True,False,False
2,63986,500,105.7,7,False,False,False,True,True,False,False,False,False
3,43026,45,94.0,5,False,False,True,False,True,False,False,True,False
4,23933,15,232.7,3,False,True,False,False,True,False,False,True,False


In [12]:
# Cell I - User features (from train only)
# - avg_rating_given, rating_count, activity_level
# - favorite_tags from high-rated history
# - default fallback for new users
# 1) Thống kê cơ bản theo user
user_stats = train_interactions.groupby("user_id").agg(
    avg_rating_given=("rating", "mean"),
    rating_count=("rating", "count"),
    min_date=("date", "min"),
    max_date=("date", "max"),
).reset_index()
# Activity level: phân tier theo số lượng ratings
user_stats["activity_level"] = pd.cut(
    user_stats["rating_count"],
    bins=[0, 5, 20, np.inf],
    labels=["cold", "warm", "active"]
)
# 2) Favorite tags: top-3 tags phổ biến nhất từ recipes user đã rating >= 4
high_rated = train_interactions[train_interactions["rating"] >= 4][["user_id", "recipe_id"]]
high_rated = high_rated.merge(
    df_recipes[["id", "tags_clean"]].rename(columns={"id": "recipe_id"}),
    on="recipe_id", how="left"
)
def get_top_tags(tags_series, top_n=3):
    from collections import Counter
    all_tags = []
    for tag_list in tags_series:
        if isinstance(tag_list, list):
            all_tags.extend(tag_list)
    if not all_tags:
        return []
    return [t for t, _ in Counter(all_tags).most_common(top_n)]
favorite_tags = high_rated.groupby("user_id")["tags_clean"].apply(get_top_tags).reset_index()
favorite_tags.columns = ["user_id", "favorite_tags"]
# 3) Merge thành user_features
user_features = user_stats.merge(favorite_tags, on="user_id", how="left")
user_features["favorite_tags"] = user_features["favorite_tags"].apply(
    lambda x: x if isinstance(x, list) else []
)
# 4) Global fallback cho user mới (không có trong train)
global_avg_rating = train_interactions["rating"].mean()
global_top_tags = get_top_tags(df_recipes["tags_clean"], top_n=3)
print(f"Global fallback: avg_rating={global_avg_rating:.2f}, top_tags={global_top_tags}")
print(f"\nuser_features shape: {user_features.shape}")
print(f"\nActivity level distribution:")
print(user_features["activity_level"].value_counts())
print(f"\nSample:")
display(user_features.head())

Global fallback: avg_rating=4.71, top_tags=['preparation', 'time-to-make', 'course']

user_features shape: (27021, 7)

Activity level distribution:
activity_level
cold      13500
warm       9021
active     4500
Name: count, dtype: int64

Sample:


,user_id,avg_rating_given,rating_count,min_date,max_date,activity_level,favorite_tags
0,1533,4.849315,73,2002-02-19,2008-03-01,active,"[time-to-make, preparation, course]"
1,1535,4.563771,541,2004-05-22,2010-08-14,active,"[preparation, time-to-make, course]"
2,1634,4.342857,35,2001-07-05,2010-02-05,active,"[time-to-make, preparation, dietary]"
3,1676,4.500000,16,2002-07-24,2010-04-23,warm,"[main-ingredient, preparation, time-to-make]"
4,1773,4.000000,3,2000-03-13,2001-01-29,cold,"[weeknight, time-to-make, course]"


In [13]:
# Cell J - Recipe Popularity Features
# Mục tiêu:
# - rating_count
# - rating_mean
# - popularity_score (Bayesian smoothing)
# - is_cold_item flag


#  1. Recipe rating statistics 
recipe_stats = (
    train_interactions
    .groupby("recipe_id")
    .agg(
        rating_count=("rating", "count"),
        rating_mean=("rating", "mean")
    )
    .reset_index()
)


#  2. Merge với danh sách recipes 
# đảm bảo mọi recipe đều có record (kể cả chưa có rating)

popularity_features = (
    df_recipes[["id"]]
    .rename(columns={"id": "recipe_id"})
    .merge(recipe_stats, on="recipe_id", how="left")
)


#  3. Fill missing values 
popularity_features["rating_count"] = popularity_features["rating_count"].fillna(0)
popularity_features["rating_mean"] = popularity_features["rating_mean"].fillna(0)


#  4. Global statistics 
global_mean_rating = train_interactions["rating"].mean()

# chọn m = percentile 75 của rating_count
m = recipe_stats["rating_count"].quantile(0.75)

print(f"Global mean rating (C): {global_mean_rating:.3f}")
print(f"Minimum votes threshold (m): {m:.1f}")


#  5. Bayesian popularity score 
v = popularity_features["rating_count"]
R = popularity_features["rating_mean"]
C = global_mean_rating

popularity_features["popularity_score"] = (
    (v / (v + m)) * R +
    (m / (v + m)) * C
)


#  6. Cold item flag 
popularity_features["is_cold_item"] = popularity_features["rating_count"] == 0


# 7. Final schema 
popularity_features = popularity_features[
    [
        "recipe_id",
        "rating_count",
        "rating_mean",
        "popularity_score",
        "is_cold_item"
    ]
]


# 8. Sanity checks 
print("\nPopularity features summary:")
print(f"Total recipes: {len(popularity_features):,}")

print(f"Cold items: {popularity_features['is_cold_item'].sum():,}")

print("\nScore distribution:")
print(popularity_features["popularity_score"].describe())


print("\nTop popular recipes:")
display(
    popularity_features
    .sort_values("popularity_score", ascending=False)
    .head()
)


print("\nSample:")
display(popularity_features.head())


Global mean rating (C): 4.714
Minimum votes threshold (m): 7.0

Popularity features summary:
Total recipes: 77,300
Cold items: 4,440

Score distribution:
count    77300.000000
mean         4.708228
std          0.129222
min          3.356865
25%          4.649806
50%          4.726919
75%          4.799611
max          4.969171
Name: popularity_score, dtype: float64

Top popular recipes:


,recipe_id,rating_count,rating_mean,popularity_score,is_cold_item
384,77497,58.0,5.000000,4.969171,False
6544,34753,54.0,5.000000,4.967149,False
39981,154351,51.0,5.000000,4.965450,False
51363,78579,165.0,4.975758,4.965094,False
70631,186029,49.0,5.000000,4.964216,False



Sample:


,recipe_id,rating_count,rating_mean,popularity_score,is_cold_item
0,137739,1.0,5.000000,4.749514,False
1,75452,4.0,4.750000,4.726919,False
2,63986,16.0,4.875000,4.825918,False
3,43026,13.0,4.923077,4.849806,False
4,23933,9.0,4.777778,4.749757,False


In [14]:
# Cell K - Feature validation checklist
# - Validate nulls, schema, key uniqueness
# - Coverage checks: users/features, recipes/vectors, recipes/top-k neighbors
# - Print final sanity summary

print("========== FEATURE VALIDATION ==========\n")


# 1. Null checks
print("1) Null checks")

print("\nrecipe_features nulls:")
print(recipe_features.isnull().sum())

print("\nuser_features nulls:")
print(user_features.isnull().sum())

print("\npopularity_features nulls:")
print(popularity_features.isnull().sum())


# 2. Key uniqueness 
print("\n2) Key uniqueness checks")

assert recipe_features["recipe_id"].is_unique, "recipe_features.recipe_id not unique"
assert user_features["user_id"].is_unique, "user_features.user_id not unique"
assert popularity_features["recipe_id"].is_unique, "popularity_features.recipe_id not unique"
print("✓ All keys are unique")


# 3. Recipe coverage
print("\n3) Recipe coverage")

n_recipes = df_recipes["id"].nunique()

print(f"Total recipes in dataset: {n_recipes:,}")
print(f"recipe_features rows:     {len(recipe_features):,}")
print(f"popularity_features rows: {len(popularity_features):,}")

assert len(recipe_features) == n_recipes, "recipe_features coverage mismatch"
assert len(popularity_features) == n_recipes, "popularity_features coverage mismatch"

print("✓ Recipe feature coverage OK")



# 4. User coverage
print("\n4) User coverage")

n_users_train = train_interactions["user_id"].nunique()

print(f"Users in train_interactions: {n_users_train:,}")
print(f"user_features rows:          {len(user_features):,}")

assert len(user_features) == n_users_train, "user_features coverage mismatch"

print("✓ User feature coverage OK")



# 5. TF-IDF vector coverage
print("\n5) TF-IDF vector coverage")

tfidf_rows = recipe_tfidf_matrix.shape[0]

print(f"Recipes: {n_recipes:,}")
print(f"TF-IDF rows: {tfidf_rows:,}")

assert tfidf_rows == n_recipes, "TF-IDF matrix row mismatch"

print("✓ TF-IDF coverage OK")


 
# 6. Recipe index map validation
print("\n6) recipe_index_map validation")

assert recipe_index_map["recipe_id"].is_unique, "recipe_index_map recipe_id not unique"

print(f"recipe_index_map rows: {len(recipe_index_map):,}")

assert len(recipe_index_map) == n_recipes, "recipe_index_map size mismatch"

print("✓ recipe_index_map valid")



# 7. Similarity coverage
print("\n7) Similarity coverage")

TOPK_NEIGHBORS = 50  # phải giống config cell A

neighbors_per_recipe = (
    recipe_similarity_topk
    .groupby("recipe_id")
    .size()
)

print("Neighbor count stats:")
print(neighbors_per_recipe.describe())

assert neighbors_per_recipe.min() == TOPK_NEIGHBORS, "Some recipes missing neighbors"

print("✓ Similarity coverage OK")



# 8. Popularity sanity checks
print("\n8) Popularity sanity checks")

print(popularity_features["popularity_score"].describe())

assert popularity_features["popularity_score"].min() >= 0
assert popularity_features["popularity_score"].max() <= 5.5

print("✓ Popularity scores valid")

 
# 9. Final summary

print("\n========== FINAL PIPELINE SUMMARY ==========\n")

print(f"Recipes:            {n_recipes:,}")
print(f"Users (train):      {n_users_train:,}")

print(f"\nTF-IDF matrix:")
print(f"  shape: {recipe_tfidf_matrix.shape}")
print(f"  nnz:   {recipe_tfidf_matrix.nnz:,}")

print(f"\nFeatures:")
print(f"  recipe_features:     {len(recipe_features):,}")
print(f"  user_features:       {len(user_features):,}")
print(f"  popularity_features: {len(popularity_features):,}")

print(f"\nSimilarity edges:")
print(f"  total rows: {len(recipe_similarity_topk):,}")

print("\n✓ All validations passed")

========== FEATURE VALIDATION ==========

1) Null checks

recipe_features nulls:
recipe_id             0
minutes               0
calories              0
n_ingredients_calc    0
is_calorie_outlier    0
is_quick_meal         0
is_medium_meal        0
is_long_meal          0
is_low_calorie        0
is_medium_calorie     0
is_high_calorie       0
is_simple_recipe      0
is_complex_recipe     0
dtype: int64

user_features nulls:
user_id             0
avg_rating_given    0
rating_count        0
min_date            0
max_date            0
activity_level      0
favorite_tags       0
dtype: int64

popularity_features nulls:
recipe_id           0
rating_count        0
rating_mean         0
popularity_score    0
is_cold_item        0
dtype: int64

2) Key uniqueness checks
✓ All keys are unique

3) Recipe coverage
Total recipes in dataset: 77,300
recipe_features rows:     77,300
popularity_features rows: 77,300
✓ Recipe feature coverage OK

4) User coverage
Users in train_interactions: 27,021
user

In [19]:
# === Cell L - Save Artifacts + Manifest ===

OUTPUT_DIR = Path("../artifacts/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fix pandas Int64 (nullable) -> int64 (numpy) for serialization
for df_fix, cols in [
    (recipe_index_map, ["recipe_id"]),
    (recipe_features, ["recipe_id"]),
    (recipe_similarity_topk, ["recipe_id", "neighbor_recipe_id"]),
    (popularity_features, ["recipe_id"]),
    (user_features, ["user_id"]),
    (train_interactions, ["user_id", "recipe_id"]),
    (val_interactions, ["user_id", "recipe_id"]),
]:
    for c in cols:
        if c in df_fix.columns:
            df_fix[c] = df_fix[c].astype("int64")

def save_df(df, path, index=False):
    """Save DataFrame: try parquet first, fall back to CSV if pyarrow fails."""
    try:
        parquet_path = path.with_suffix(".parquet")
        df.to_parquet(parquet_path, index=index)
        return parquet_path
    except Exception:
        csv_path = path.with_suffix(".csv")
        df.to_csv(csv_path, index=index)
        return csv_path

# 1) TF-IDF vectorizer
joblib.dump(tfidf_vectorizer, OUTPUT_DIR / "tfidf_vectorizer.joblib")

# 2) TF-IDF sparse matrix
sparse.save_npz(OUTPUT_DIR / "recipe_tfidf_matrix.npz", recipe_tfidf_matrix)

# 3) Recipe index map
save_df(recipe_index_map, OUTPUT_DIR / "recipe_index_map")

# 4) Similarity top-k
save_df(recipe_similarity_topk, OUTPUT_DIR / "recipe_similarity_topk")

# 5) Recipe features
save_df(recipe_features, OUTPUT_DIR / "recipe_features")

# 6) User features -- favorite_tags là list, convert sang str trước khi lưu
user_features_out = user_features.copy()
user_features_out["favorite_tags"] = user_features_out["favorite_tags"].apply(str)
save_df(user_features_out, OUTPUT_DIR / "user_features")

# 7) Popularity features
save_df(popularity_features, OUTPUT_DIR / "popularity_features")

# 8) Train/Val interactions
save_df(train_interactions, OUTPUT_DIR / "train_interactions")
save_df(val_interactions, OUTPUT_DIR / "val_interactions")

# 9) Feature manifest -- scan actual saved files
saved_files = sorted([f.name for f in OUTPUT_DIR.iterdir() if f.is_file() and f.name != "feature_manifest.json"])

manifest = {
    "version": "1.0",
    "build_timestamp": datetime.now().isoformat(),
    "source_data": {
        "recipes_clean": str(Path("../data/processed/recipes_clean.csv").resolve()),
        "interactions_clean": str(Path("../data/processed/interactions_clean.csv").resolve()),
    },
    "stats": {
        "n_recipes": int(len(recipe_features)),
        "n_users_train": int(len(user_features)),
        "n_train_interactions": int(len(train_interactions)),
        "n_val_interactions": int(len(val_interactions)),
        "tfidf_shape": list(recipe_tfidf_matrix.shape),
        "similarity_topk": int(recipe_similarity_topk.groupby("recipe_id").size().iloc[0]),
        "similarity_rows": int(len(recipe_similarity_topk)),
    },
    "config": {
        "tfidf_max_features": 20000,
        "tfidf_ngram_range": [1, 2],
        "tfidf_min_df": 5,
        "tfidf_max_df": 0.95,
        "similarity_top_k": 50,
        "train_val_split_ratio": 0.8,
        "split_method": "time-based",
    },
    "artifacts": saved_files,
}

with open(OUTPUT_DIR / "feature_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# ---------- Print summary ----------
print("Artifacts saved to:", OUTPUT_DIR.resolve())
print()
for f_name in saved_files + ["feature_manifest.json"]:
    p = OUTPUT_DIR / f_name
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {f_name:<45s} {size_mb:>8.2f} MB")
print(f"\nBuild timestamp: {manifest['build_timestamp']}")
print("\nNotebook 03 COMPLETE.")

Artifacts saved to: D:\GITHUB\Food-RecommendationSystem\artifacts\features

  popularity_features.csv                           3.24 MB
  recipe_features.csv                               5.22 MB
  recipe_index_map.csv                              1.00 MB
  recipe_similarity_topk.csv                      124.28 MB
  recipe_tfidf_matrix.npz                          70.46 MB
  tfidf_vectorizer.joblib                           0.80 MB
  train_interactions.csv                          164.66 MB
  user_features.csv                                 2.39 MB
  val_interactions.csv                             42.54 MB
  feature_manifest.json                             0.00 MB

Build timestamp: 2026-04-17T15:02:37.909049

Notebook 03 COMPLETE.
